# Hyperparameter Sweep Visualization

This notebook demonstrates the flexible hyperparameter visualization tools.

## Key Features:
- **Mean with 95% bootstrapped confidence intervals** (instead of IQM)
- **Flexible grouping** by one or more hyperparameters (color-coded)
- **Algorithm-agnostic** - works with IPPO, SAC, TD3, etc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional, Union

# Import our visualization module (adjust path as needed)
# Option 1: If module is in same directory or PYTHONPATH
# from hparam_visualizations import *

# Option 2: Add to path if needed
import sys
sys.path.insert(0, '.')  # or '/path/to/module/directory'

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## Core Functions (Inline Version)

Below is the complete implementation you can use directly in your notebook:

In [ ]:
import os
from dataclasses import dataclass
from itertools import product

# =============================================================================
# Data Loading
# =============================================================================

def load_npy_files(directory: str) -> Dict[str, np.ndarray]:
    """Load all .npy files from a directory into a dictionary."""
    npy_dict = {}
    for file in os.listdir(directory):
        if file.endswith('.npy'):
            key = file.replace('.npy', '')
            file_path = os.path.join(directory, file)
            npy_dict[key] = np.load(file_path)
    return npy_dict


# =============================================================================
# Statistical Functions
# =============================================================================

def bootstrap_ci_mean(
    data: np.ndarray,
    n_bootstrap: int = 1000,
    confidence_level: float = 0.95,
    random_seed: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Calculate bootstrapped confidence intervals for the mean.
    
    Args:
        data: numpy array of shape [n_seeds, n_points]
        n_bootstrap: number of bootstrap samples
        confidence_level: confidence level (default 0.95 for 95% CI)
        random_seed: optional random seed for reproducibility
        
    Returns:
        Tuple of (mean, lower_ci, upper_ci)
    """
    if random_seed is not None:
        np.random.seed(random_seed)
    
    n_seeds, n_points = data.shape
    bootstrap_means = np.zeros((n_bootstrap, n_points))
    
    for i in range(n_bootstrap):
        seed_indices = np.random.randint(0, n_seeds, size=n_seeds)
        bootstrap_sample = data[seed_indices]
        bootstrap_means[i] = np.mean(bootstrap_sample, axis=0)
    
    mean = np.mean(data, axis=0)
    alpha = (1 - confidence_level) / 2
    lower_ci = np.percentile(bootstrap_means, 100 * alpha, axis=0)
    upper_ci = np.percentile(bootstrap_means, 100 * (1 - alpha), axis=0)
    
    return mean, lower_ci, upper_ci


def compute_auc(returns: np.ndarray, axis: int = -1) -> np.ndarray:
    """Compute Area Under Curve (mean over timesteps)."""
    return np.mean(returns, axis=axis)


# =============================================================================
# Grouping Utilities
# =============================================================================

@dataclass
class GroupInfo:
    """Information about a group of configurations."""
    name: str
    indices: np.ndarray
    color: str
    values: Dict


def create_groups(
    hp_dict: Dict[str, np.ndarray],
    group_by: Union[str, List[str]]
) -> Tuple[List[GroupInfo], Dict[str, List]]:
    """
    Create groups based on one or more hyperparameters.
    
    Args:
        hp_dict: Dictionary of hyperparameters
        group_by: Single param name or list of param names to group by
        
    Returns:
        List of GroupInfo objects and dict of unique values per param
        
    Example:
        group_by=['update_epochs', 'num_minibatches']
        -> Groups: (2,2), (2,4), (4,2), (4,4) etc.
    """
    if isinstance(group_by, str):
        group_by = [group_by]
    
    # Get unique values for each grouping parameter
    unique_values = {}
    for param in group_by:
        if param not in hp_dict:
            raise ValueError(f"Parameter '{param}' not found. Available: {list(hp_dict.keys())}")
        unique_values[param] = sorted(list(set(hp_dict[param])))
    
    # Generate all combinations
    all_combinations = list(product(*[unique_values[p] for p in group_by]))
    n_groups = len(all_combinations)
    colors = sns.color_palette("husl", n_groups)
    
    groups = []
    for combo_idx, combo in enumerate(all_combinations):
        mask = np.ones(len(hp_dict[group_by[0]]), dtype=bool)
        values_dict = {}
        
        for param, value in zip(group_by, combo):
            mask &= (np.array(hp_dict[param]) == value)
            values_dict[param] = value
        
        indices = np.where(mask)[0]
        
        if len(indices) > 0:
            if len(group_by) == 1:
                name = f"{group_by[0]}={combo[0]}"
            else:
                name = ", ".join([f"{p}={v}" for p, v in zip(group_by, combo)])
            
            groups.append(GroupInfo(
                name=name,
                indices=indices,
                color=colors[combo_idx],
                values=values_dict
            ))
    
    return groups, unique_values


def format_hparam_value(value, param_name: str = "") -> str:
    """Format hyperparameter value for display."""
    if 'lr' in param_name.lower() or 'learning' in param_name.lower():
        return f"{value:.2e}"
    elif isinstance(value, float) and abs(value) < 0.1:
        return f"{value:.4f}"
    elif isinstance(value, float):
        return f"{value:.3f}"
    elif isinstance(value, (int, np.integer)):
        return str(int(value))
    return str(value)

In [ ]:
# =============================================================================
# Algorithm Configurations
# =============================================================================

ALGORITHM_CONFIGS = {
    'ippo': {
        'params': ['lr', 'clip_eps', 'ent_coef'],
        'display_names': ['Learning Rate', 'Clip Epsilon', 'Entropy Coef'],
        'log_scale': [True, False, False],
    },
    'ppo': {
        'params': ['lr', 'clip_eps', 'ent_coef'],
        'display_names': ['Learning Rate', 'Clip Epsilon', 'Entropy Coef'],
        'log_scale': [True, False, False],
    },
    'sac': {
        'params': ['p_lr', 'q_lr', 'alpha_lr', 'tau'],
        'display_names': ['Policy LR', 'Q LR', 'Alpha LR', 'Tau'],
        'log_scale': [True, True, True, False],
    },
    'isac': {
        'params': ['p_lr', 'q_lr', 'alpha_lr', 'tau'],
        'display_names': ['Policy LR', 'Q LR', 'Alpha LR', 'Tau'],
        'log_scale': [True, True, True, False],
    },
    'td3': {
        'params': ['actor_lr', 'critic_lr', 'tau'],
        'display_names': ['Actor LR', 'Critic LR', 'Tau'],
        'log_scale': [True, True, False],
    },
}


def get_algorithm_config(
    algorithm: str,
    custom_params: Optional[List[str]] = None,
    custom_display_names: Optional[List[str]] = None,
    custom_log_scale: Optional[List[bool]] = None
) -> Dict:
    """Get or create algorithm configuration."""
    if algorithm.lower() in ALGORITHM_CONFIGS:
        config = ALGORITHM_CONFIGS[algorithm.lower()].copy()
    else:
        config = {'params': [], 'display_names': [], 'log_scale': []}
    
    if custom_params is not None:
        config['params'] = custom_params
        if custom_display_names is None:
            config['display_names'] = [p.replace('_', ' ').title() for p in custom_params]
        else:
            config['display_names'] = custom_display_names
        if custom_log_scale is None:
            config['log_scale'] = ['lr' in p.lower() for p in custom_params]
        else:
            config['log_scale'] = custom_log_scale
    elif custom_display_names is not None:
        config['display_names'] = custom_display_names
    
    if custom_log_scale is not None:
        config['log_scale'] = custom_log_scale
    
    return config

In [ ]:
# =============================================================================
# Main Plotting Functions
# =============================================================================

def plot_training_curves(
    returns: np.ndarray,
    checkpoint_steps: Optional[np.ndarray] = None,
    labels: Optional[List[str]] = None,
    title: str = "Training Returns",
    xlabel: str = "Steps",
    ylabel: str = "Mean Episode Return",
    n_bootstrap: int = 1000,
    confidence_level: float = 0.95,
    figsize: Tuple[int, int] = (12, 8),
    random_seed: Optional[int] = None
) -> plt.Figure:
    """
    Plot training returns with mean and 95% CI.
    
    Args:
        returns: [n_hyperparams, n_seeds, n_steps]
    """
    if checkpoint_steps is None:
        checkpoint_steps = np.arange(returns.shape[2])
    
    n_hyperparams = returns.shape[0]
    if labels is None:
        labels = [f"Config {i+1}" for i in range(n_hyperparams)]
    
    fig, ax = plt.subplots(figsize=figsize)
    sns.set_style("whitegrid")
    colors = sns.color_palette("husl", n_hyperparams)
    
    for param_idx in range(n_hyperparams):
        data = returns[param_idx]
        mean, lower_ci, upper_ci = bootstrap_ci_mean(
            data, n_bootstrap=n_bootstrap, 
            confidence_level=confidence_level, 
            random_seed=random_seed
        )
        ax.plot(checkpoint_steps, mean, label=labels[param_idx], color=colors[param_idx])
        ax.fill_between(checkpoint_steps, lower_ci, upper_ci, alpha=0.2, color=colors[param_idx])
    
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    return fig


def plot_auc_scatter(
    returns: np.ndarray,
    hp_dict: Dict[str, np.ndarray],
    params_to_plot: List[str],
    group_by: Optional[Union[str, List[str]]] = None,
    display_names: Optional[List[str]] = None,
    log_scale: Optional[List[bool]] = None,
    title: str = "AUC vs Hyperparameters",
    figsize_per_subplot: Tuple[float, float] = (5, 4),
    alpha: float = 0.6,
    marker_size: int = 50
) -> plt.Figure:
    """
    Create scatter plots of AUC vs hyperparameters with optional grouping.
    
    Args:
        returns: [n_configs, n_seeds, n_steps]
        hp_dict: Dictionary of hyperparameters
        params_to_plot: List of param names for subplots
        group_by: Param(s) to group by for coloring
        display_names: Human-readable names for params
        log_scale: Whether to use log scale for x-axis
        
    Example:
        # IPPO with grouping by update_epochs and num_minibatches
        plot_auc_scatter(returns, hp_dict, 
                        params_to_plot=['lr', 'clip_eps', 'ent_coef'],
                        group_by=['update_epochs', 'num_minibatches'])
    """
    # Validate
    for param in params_to_plot:
        if param not in hp_dict:
            raise ValueError(f"'{param}' not in hp_dict. Available: {list(hp_dict.keys())}")
    
    n_params = len(params_to_plot)
    if display_names is None:
        display_names = [p.replace('_', ' ').title() for p in params_to_plot]
    if log_scale is None:
        log_scale = ['lr' in p.lower() for p in params_to_plot]
    
    # Compute AUC
    auc_returns = compute_auc(returns, axis=2)  # [n_configs, n_seeds]
    
    # Create groups
    if group_by is not None:
        groups, unique_vals = create_groups(hp_dict, group_by)
    else:
        groups = [GroupInfo(
            name='All',
            indices=np.arange(len(hp_dict[params_to_plot[0]])),
            color='blue',
            values={}
        )]
    
    # Create figure
    fig, axes = plt.subplots(1, n_params, 
                             figsize=(figsize_per_subplot[0] * n_params, figsize_per_subplot[1]))
    if n_params == 1:
        axes = [axes]
    
    # Plot each parameter
    for ax, param, display_name, use_log in zip(axes, params_to_plot, display_names, log_scale):
        param_values = np.array(hp_dict[param])
        
        for group in groups:
            for idx in group.indices:
                x_val = param_values[idx]
                if use_log:
                    x_val = np.log10(x_val)
                
                y_vals = auc_returns[idx]
                ax.scatter(
                    [x_val] * len(y_vals),
                    y_vals,
                    alpha=alpha,
                    color=group.color,
                    s=marker_size,
                    label=group.name if idx == group.indices[0] else None
                )
        
        xlabel = f"log10({display_name})" if use_log else display_name
        ax.set_xlabel(xlabel)
        ax.set_ylabel('AUC')
        ax.set_title(display_name)
        ax.grid(True, alpha=0.3)
    
    # Legend
    if group_by is not None:
        handles, labels_legend = axes[-1].get_legend_handles_labels()
        by_label = dict(zip(labels_legend, handles))
        fig.legend(by_label.values(), by_label.keys(), 
                  loc='center right', bbox_to_anchor=(1.15, 0.5))
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig


def plot_auc_scatter_algorithm(
    returns: np.ndarray,
    hp_dict: Dict[str, np.ndarray],
    algorithm: str,
    group_by: Optional[Union[str, List[str]]] = None,
    custom_params: Optional[List[str]] = None,
    custom_display_names: Optional[List[str]] = None,
    custom_log_scale: Optional[List[bool]] = None,
    title: Optional[str] = None,
    **kwargs
) -> plt.Figure:
    """
    Convenience function for algorithm-specific AUC scatter plots.
    
    Supported algorithms: 'ippo', 'ppo', 'sac', 'isac', 'td3'
    """
    config = get_algorithm_config(
        algorithm,
        custom_params=custom_params,
        custom_display_names=custom_display_names,
        custom_log_scale=custom_log_scale
    )
    
    # Filter to params that exist
    valid_params = []
    valid_display_names = []
    valid_log_scale = []
    
    for param, display, log in zip(config['params'], config['display_names'], config['log_scale']):
        if param in hp_dict:
            valid_params.append(param)
            valid_display_names.append(display)
            valid_log_scale.append(log)
        else:
            print(f"Warning: '{param}' not found, skipping.")
    
    if not valid_params:
        raise ValueError(f"No valid params for '{algorithm}'. Available: {list(hp_dict.keys())}")
    
    if title is None:
        title = f"{algorithm.upper()} - AUC vs Hyperparameters"
    
    return plot_auc_scatter(
        returns, hp_dict,
        params_to_plot=valid_params,
        group_by=group_by,
        display_names=valid_display_names,
        log_scale=valid_log_scale,
        title=title,
        **kwargs
    )

In [ ]:
# =============================================================================
# Analysis Functions
# =============================================================================

def analyze_performance(
    returns: np.ndarray,
    hp_dict: Dict[str, np.ndarray],
    window_size: int = 10
) -> Tuple[Dict, np.ndarray]:
    """Analyze performance metrics for each configuration."""
    n_configs, n_seeds, n_steps = returns.shape
    final_performances = np.zeros(n_configs)
    performance_metrics = {}
    
    for config_idx in range(n_configs):
        data = returns[config_idx]
        mean_curve = np.mean(data, axis=0)
        final_performance = np.mean(mean_curve[-window_size:])
        final_performances[config_idx] = final_performance
        auc = np.mean(mean_curve)
        
        performance_metrics[config_idx] = {
            'config_idx': config_idx,
            'hyperparams': {k: v[config_idx] for k, v in hp_dict.items()},
            'final_performance': final_performance,
            'auc': auc,
            'max_performance': np.max(mean_curve),
            'std_across_seeds': np.std(np.mean(data, axis=1))
        }
    
    return performance_metrics, final_performances


def print_best_configs(
    performance_metrics: Dict,
    final_performances: np.ndarray,
    n_best: int = 5,
    metric: str = 'final_performance'
) -> None:
    """Print best performing configurations."""
    if metric == 'final_performance':
        values = final_performances
    else:
        values = np.array([performance_metrics[i][metric] for i in range(len(performance_metrics))])
    
    best_indices = np.argsort(values)[-n_best:][::-1]
    
    print(f"\nTop {n_best} Configurations (by {metric}):")
    print("=" * 70)
    
    for rank, idx in enumerate(best_indices, 1):
        metrics = performance_metrics[idx]
        print(f"\nRank {rank} (Config {metrics['config_idx'] + 1})")
        print(f"  Final: {metrics['final_performance']:.2f} | AUC: {metrics['auc']:.2f} | Max: {metrics['max_performance']:.2f}")
        print("  Hyperparams:", end="")
        for param, value in metrics['hyperparams'].items():
            print(f" {param}={format_hparam_value(value, param)}", end="")
        print()

---
## Example Usage with Synthetic Data

Let's create some synthetic data to demonstrate the visualization tools.

In [ ]:
# Create synthetic data for demonstration
np.random.seed(42)

n_configs = 24  # e.g., 4 lr x 3 clip_eps x 2 update_epochs
n_seeds = 6
n_steps = 100

# Generate synthetic returns (random walk with trend)
returns = np.random.randn(n_configs, n_seeds, n_steps).cumsum(axis=2)
returns = returns + np.linspace(0, 100, n_steps)  # Add upward trend
returns = returns + np.random.randn(n_configs, 1, 1) * 20  # Add config-level variance

# Generate synthetic hyperparameters
# Simulating a grid search over: lr (4) x clip_eps (3) x update_epochs (2) = 24 configs
lrs = [1e-4, 2.5e-4, 5e-4, 1e-3]
clip_epss = [0.1, 0.2, 0.3]
update_epochs_vals = [2, 4]

hp_dict = {
    'lr': [],
    'clip_eps': [],
    'ent_coef': [],
    'update_epochs': [],
    'num_minibatches': []
}

for ue in update_epochs_vals:
    for clip in clip_epss:
        for lr in lrs:
            hp_dict['lr'].append(lr)
            hp_dict['clip_eps'].append(clip)
            hp_dict['ent_coef'].append(np.random.choice([0.001, 0.01]))
            hp_dict['update_epochs'].append(ue)
            hp_dict['num_minibatches'].append(np.random.choice([2, 4]))

# Convert to numpy arrays
for key in hp_dict:
    hp_dict[key] = np.array(hp_dict[key])

print(f"Returns shape: {returns.shape}")
print(f"Number of configs: {n_configs}")
print(f"Hyperparameters: {list(hp_dict.keys())}")

### Example 1: IPPO AUC Plot - Grouped by `update_epochs`

In [ ]:
fig = plot_auc_scatter_algorithm(
    returns, 
    hp_dict, 
    algorithm='ippo',
    group_by='update_epochs',
    title='IPPO - AUC vs Hyperparameters (grouped by update_epochs)'
)
plt.show()

### Example 2: IPPO AUC Plot - Grouped by Combination of `update_epochs` AND `num_minibatches`

This is the key feature you requested - grouping by multiple hyperparameters creates combinations like (2, 2), (2, 4), (4, 2), (4, 4).

In [ ]:
fig = plot_auc_scatter_algorithm(
    returns, 
    hp_dict, 
    algorithm='ippo',
    group_by=['update_epochs', 'num_minibatches'],  # <-- Multiple grouping params!
    title='IPPO - AUC grouped by (update_epochs, num_minibatches)'
)
plt.show()

### Example 3: Custom Parameters (Flexible API)

In [ ]:
# Use the generic plot_auc_scatter for full flexibility
fig = plot_auc_scatter(
    returns, 
    hp_dict,
    params_to_plot=['lr', 'clip_eps'],  # Only plot these two
    group_by='ent_coef',
    display_names=['Policy Learning Rate', 'Clipping Epsilon'],
    log_scale=[True, False],  # Log scale for LR, linear for clip_eps
    title='Custom 2-param Plot'
)
plt.show()

### Example 4: SAC-style Plot (Different Algorithm Family)

Let's create synthetic SAC data to show algorithm flexibility.

In [ ]:
# Synthetic SAC hyperparameters
n_sac_configs = 20
sac_returns = np.random.randn(n_sac_configs, n_seeds, n_steps).cumsum(axis=2)
sac_returns = sac_returns + np.linspace(0, 150, n_steps)

sac_hp_dict = {
    'p_lr': np.random.choice([1e-4, 3e-4, 1e-3], n_sac_configs),
    'q_lr': np.random.choice([1e-4, 3e-4, 1e-3], n_sac_configs),
    'alpha_lr': np.random.choice([1e-4, 3e-4], n_sac_configs),
    'tau': np.random.choice([0.005, 0.01, 0.02], n_sac_configs),
    'buffer_size': np.random.choice([100000, 1000000], n_sac_configs),
}

# SAC plot with all 4 default params
fig = plot_auc_scatter_algorithm(
    sac_returns,
    sac_hp_dict,
    algorithm='sac',
    group_by='buffer_size',
    title='SAC - AUC vs Learning Rates and Tau'
)
plt.show()

### Example 5: SAC with only 3 parameters (skip alpha_lr)

In [ ]:
# Use custom_params to select which params to plot
fig = plot_auc_scatter_algorithm(
    sac_returns,
    sac_hp_dict,
    algorithm='sac',
    custom_params=['p_lr', 'q_lr', 'tau'],  # Skip alpha_lr
    group_by='buffer_size',
    title='SAC - Policy LR, Q LR, Tau only'
)
plt.show()

### Example 6: Training Curves with Mean + 95% CI

In [ ]:
# Select a few configs to plot training curves
selected_configs = [0, 5, 10, 15]  # Arbitrary selection
selected_returns = returns[selected_configs]

labels = [f"lr={hp_dict['lr'][i]:.1e}, clip={hp_dict['clip_eps'][i]:.1f}" 
          for i in selected_configs]

fig = plot_training_curves(
    selected_returns,
    labels=labels,
    title='Training Curves (Mean ± 95% CI)',
    ylabel='Mean Episode Return',
    n_bootstrap=1000,
    confidence_level=0.95
)
plt.show()

### Example 7: Best Configuration Analysis

In [ ]:
metrics, final_perfs = analyze_performance(returns, hp_dict)
print_best_configs(metrics, final_perfs, n_best=5, metric='auc')

---
## Loading Real Data (Template)

Here's how to load your actual sweep data:

In [ ]:
# Uncomment and modify paths for your data

# # Load IPPO FF data
# ippo_ff_returns = np.load("/path/to/IPPO/scratchitch/FF/metric/eval_returns.npy")
# ippo_ff_hp_dict = load_npy_files("/path/to/IPPO/scratchitch/FF/hparam")

# print(f"Returns shape: {ippo_ff_returns.shape}")
# print(f"Available hyperparameters: {list(ippo_ff_hp_dict.keys())}")

# # IPPO plot grouped by update_epochs
# fig = plot_auc_scatter_algorithm(
#     ippo_ff_returns,
#     ippo_ff_hp_dict,
#     algorithm='ippo',
#     group_by='update_epochs',
#     title='IPPO FF - Scratchitch'
# )
# plt.show()

# # Or grouped by both update_epochs AND num_minibatches
# fig = plot_auc_scatter_algorithm(
#     ippo_ff_returns,
#     ippo_ff_hp_dict,
#     algorithm='ippo',
#     group_by=['update_epochs', 'num_minibatches'],
#     title='IPPO FF - Scratchitch (grouped by epochs, minibatches)'
# )
# plt.show()

---
## Summary of Key Functions

| Function | Purpose |
|----------|--------|
| `plot_auc_scatter()` | Generic scatter plots - full flexibility |
| `plot_auc_scatter_algorithm()` | Algorithm-specific presets (ippo, sac, td3, etc.) |
| `plot_training_curves()` | Training curves with mean + 95% CI |
| `analyze_performance()` | Compute metrics for all configs |
| `print_best_configs()` | Display top N configurations |
| `create_groups()` | Create color groups from hyperparams |
| `bootstrap_ci_mean()` | Compute bootstrapped 95% CI for mean |